In [ ]:
# https://ai.google.dev/gemini-api/docs/live-api/get-started-sdk#python_1
# https://googleapis.github.io/python-genai/genai.html#module-genai.live

In [120]:
import asyncio
import os
import urllib.request

import numpy as np
import sounddevice as sd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Audio

load_dotenv()

True

# Microphone capture test

In [96]:
SAMPLE_RATE = 16_000 # audio frames per second

duration_secs = 5
audio = sd.rec(
    frames= int(duration_secs * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype=np.int16,
)

sd.wait()

In [97]:
Audio(audio.reshape(-1), rate=SAMPLE_RATE)

In [98]:
print(audio.shape[0], duration_secs * SAMPLE_RATE)

audio_bytes = audio.reshape(-1).astype("<i2").tobytes()
print(len(audio_bytes), audio.shape[0] * 2) # as each element is 2 bytes

80000 80000
160000 160000


# Gemini Live Session Connection

In [131]:
test_audio = urllib.request.urlopen(
    "https://storage.googleapis.com/generativeai-downloads/data/hello_are_you_there.pcm"
).read()

print("Bytes:", len(test_audio))

Audio(np.frombuffer(test_audio, dtype="<i2"), rate=16e3)

Bytes: 96938


In [155]:
response_audio = bytearray()
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))


model = "gemini-3.8-live"
config = {
    "response_modalities": ["AUDIO"],
}

padded_audio = test_audio + b"\x00\x00" * SAMPLE_RATE


async def main():
    async with client.aio.live.connect(model=model, config=config) as session:
        print("Session established")

        await session.send_realtime_input(audio=types.Blob(data=padded_audio, mime_type="audio/pcm;rate=16000"))
        await session.send_realtime_input(audio_stream_end=True)
        print("Ended audio stream..")

        async for response in session.receive():
            content = response.server_content
            if content:
                if content.model_turn:
                    for part in content.model_turn.parts:
                        data = part.inline_data.data
                        response_audio.extend(data)

                if content.input_transcription:
                    print(f"User: {content.input_transcription.text}")
                if content.output_transcription:
                    print(f"Model: {content.output_transcription.text}")


await main()

Session established
Ended audio stream..
User: Hey, can you hear me?
Model: Yes, I can 
Model: hear you perfectly!
Model:  How can I help you today?


In [156]:
response_np = np.frombuffer(response_audio, dtype="<i2")
sd.play(response_np, samplerate=24_000)